In [0]:
# If needed (run once per cluster)
# %pip install -U databricks-vectorsearch
# dbutils.library.restartPython()

import time
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

endpoint_name = "healthbot-vs-endpoint"
endpoint_type = "STANDARD"   # or "STORAGE_OPTIMIZED"

# 1) Create endpoint
try:
    client.create_endpoint(
        name=endpoint_name,
        endpoint_type=endpoint_type
    )
    print(f"Endpoint creation requested: {endpoint_name}")
except Exception as e:
    # If it already exists, Databricks throws an error – safe to continue
    print(f"Create endpoint returned: {e}")

# 2) Wait until endpoint is READY
def wait_for_endpoint_ready(
    client,
    endpoint_name,
    poll_seconds=15,
    timeout_seconds=1800   # up to 30 min
):
    start = time.time()
    while True:
        ep = client.get_endpoint(endpoint_name)
        state = ep.get('endpoint_status', {}).get('state', '') 

        print(f"Endpoint {endpoint_name} status: {state}")

        if state == "ONLINE":
            print("✅ Endpoint is READY")
            return ep

        if time.time() - start > timeout_seconds:
            raise TimeoutError(
                f"Endpoint {endpoint_name} not ready after {timeout_seconds}s"
            )

        time.sleep(poll_seconds)

endpoint_info = wait_for_endpoint_ready(client, endpoint_name)

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
client.get_endpoint("healthbot-vs-endpoint")
